# Ground truth vs prediction — map checks

Compare OM4 (ground truth) to a Samudra `predictions.zarr` on the same time span: **SST**, **SSH** (`zos`), **surface `uo` / `vo`**, **surface `so`**, and an optional **depth** slice of $\theta_O$.

Edit the paths in the next cell. Uses `utils.notebook.process_data` (same pipeline as `samudra_plotting.ipynb`). For rollouts shorter than 600 steps, time coordinates are **intersected** automatically.


In [ ]:
import sys
from pathlib import Path

import cmocean as cm
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.ticker import FixedLocator
from dask.diagnostics import ProgressBar

# Add ../src (local) or /opt/samudra/src (Jupyter pod with root_dir=/opt/samudra)
for _root in (Path.cwd(), Path.cwd().parent, Path("/opt/samudra")):
    if (_root / "src" / "utils" / "notebook.py").is_file():
        sys.path.insert(0, str(_root / "src"))
        break
else:
    raise FileNotFoundError("Could not find src/utils/notebook.py; set cwd or clone repo under /opt/samudra")

from utils.notebook import process_data

### Paths and labels

- `DATA_ZARR`: OM4 data store (directory with `.zmetadata`), same as training/rollout `data_dir/data`.
- `PREDICTIONS_ZARR`: rollout `predictions.zarr` directory.
- `FIGURE_DIR` / `SAVE_FIGURES` / `FIG_DPI`: PNG output (e.g. on PVC: `/data/samudra/rollout/<run>/gt_vs_pred_maps`). Files: `sst_mean_maps`, `ssh_mean_maps`, `uo_surface_mean_maps`, `vo_surface_mean_maps`, `so_surface_mean_maps`, `theta_lev*`.

In [ ]:
from pathlib import Path

# --- edit these ---
DATA_ZARR = Path("/data/samudra/data")
PREDICTIONS_ZARR = Path(
    "/data/samudra/rollout/2026-03-25-rollout-om4_samudra-samudra-rollout-om4-p96pb/predictions.zarr"
)
MODEL_NAME = "Samudra"

# Initial ground-truth time slice (intersected with prediction times inside process_data)
TIME_START = "2014-10-10"
TIME_END = "2022-12-24"

pred_dict = {
    "pred_1": {
        "mode": "thermo_dynamic",
        "name": MODEL_NAME,
        "path": str(PREDICTIONS_ZARR.resolve()),
        "ls": ["uo", "vo", "thetao", "so", "zos"],
    },
}

# PNG exports (e.g. PVC: Path("/data/samudra/rollout/<run>/gt_vs_pred_maps"))
FIGURE_DIR = Path("../results/gt_vs_pred_maps_figs")
SAVE_FIGURES = True
FIG_DPI = 150

### Load and align datasets

**Kubernetes Jupyter:** set paths to the PVC, e.g. `DATA_ZARR = Path('/data/samudra/data')` and `PREDICTIONS_ZARR = Path('/data/samudra/rollout/<run>/predictions.zarr')`.


In [ ]:
# OM4 uses cftime on `time`; rollout predictions.zarr uses step indices 0..N-1 (int or float in zarr).
# `align_times=True` pairs the first N OM4 times with the first N prediction steps (see process_data).
train_data = xr.open_dataset(
    DATA_ZARR,
    engine="zarr",
    chunks={"time": 10, "lat": 180, "lon": 360},
)
train_data = train_data.sel(time=slice(TIME_START, TIME_END))

ds_gt, pred_dict = process_data(
    train_data,
    pred_dict,
    align_times=True,
    require_rollout_length_600=False,
)
ds_pr = pred_dict["pred_1"]["ds_prediction"]
print("Ground truth time steps:", ds_gt.sizes.get("time"))
print("Prediction time steps:", ds_pr.sizes.get("time"))

### Helpers (Cartopy maps, same style as paper notebook)

In [ ]:
def _land_mask_surface(ds):
    return np.isnan(ds["thetao"]).isel(lev=0).isel(time=min(5, ds.sizes["time"] - 1))


def mean_sst(ds):
    m = _land_mask_surface(ds)
    sst = ds["thetao"].isel(lev=0).mean("time").where(~m)
    sst = sst.rename("SST")
    sst.attrs["units"] = "°C"
    return sst


def _surface_zos(ds):
    z = ds["zos"]
    if "lev" in z.dims:
        z = z.isel(lev=0)
    return z


def mean_zos(ds):
    z = _surface_zos(ds)
    m = np.isnan(z).isel(time=min(5, ds.sizes["time"] - 1))
    z = z.mean("time").where(~m)
    z = z.rename("SSH")
    z.attrs["units"] = "m"
    return z


def mean_surface_field(ds, var: str, long_name: str, units: str):
    """Time-mean at surface (`lev=0` if present), masked by ocean."""
    m = _land_mask_surface(ds)
    v = ds[var]
    if "lev" in v.dims:
        v = v.isel(lev=0)
    out = v.mean("time").where(~m)
    out = out.rename(long_name)
    out.attrs["units"] = units
    return out


def plot_scalar_map(
    da, ax, title, cmap, vmin=None, vmax=None, robust=True, col=0
):
    colormap = cmap
    colormap.set_bad(color=(0.7, 0.7, 0.7, 0))
    if vmin is None and vmax is None and robust:
        vmin = float(np.nanpercentile(da.values, 2))
        vmax = float(np.nanpercentile(da.values, 98))
    im = ax.pcolormesh(
        da["x"],
        da["y"],
        da,
        shading="auto",
        cmap=colormap,
        transform=ccrs.PlateCarree(),
        vmin=vmin,
        vmax=vmax,
    )
    ax.add_feature(cfeature.COASTLINE, edgecolor="black")
    ax.set_title(title, fontsize=12)
    gl = ax.gridlines(draw_labels=True, color="0.4", linestyle="--", alpha=0)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlocator = FixedLocator([-120, -60, 0, 60, 120])
    if col > 0:
        gl.left_labels = False
    return im


def three_panel_maps(gt_da, pr_da, label_gt, label_pr, cmap, diff_cmap=None, suptitle=None):
    diff = pr_da - gt_da
    fig, axs = plt.subplots(
        1,
        3,
        figsize=(14, 4),
        subplot_kw={"projection": ccrs.PlateCarree()},
        constrained_layout=True,
    )
    stacked = np.concatenate([gt_da.values.ravel(), pr_da.values.ravel()])
    vmin = float(np.nanpercentile(stacked, 2))
    vmax = float(np.nanpercentile(stacked, 98))
    im0 = plot_scalar_map(
        gt_da, axs[0], label_gt, cmap, vmin=vmin, vmax=vmax, robust=False, col=0
    )
    im1 = plot_scalar_map(
        pr_da, axs[1], label_pr, cmap, vmin=vmin, vmax=vmax, robust=False, col=1
    )
    dmin = float(np.nanpercentile(diff.values, 2))
    dmax = float(np.nanpercentile(diff.values, 98))
    lim = max(abs(dmin), abs(dmax))
    if lim == 0:
        lim = 1e-6
    diff_cmap = diff_cmap or cm.cm.balance
    im2 = plot_scalar_map(
        diff,
        axs[2],
        f"{label_pr} − {label_gt}",
        diff_cmap,
        vmin=-lim,
        vmax=lim,
        robust=False,
        col=2,
    )
    fig.colorbar(im0, ax=axs[:2], orientation="vertical", fraction=0.025, pad=0.02)
    fig.colorbar(im2, ax=axs[2], orientation="vertical", fraction=0.06, pad=0.02)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    return fig, axs


def save_figure(fig, stem: str) -> None:
    """Write ``{stem}.png`` under ``FIGURE_DIR`` when ``SAVE_FIGURES`` is True."""
    if not SAVE_FIGURES:
        return
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    out = FIGURE_DIR / f"{stem}.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    print(f"Saved {out.resolve()}")


### SST (surface potential temperature, mean over time)

Run with a progress bar while Dask computes.

In [ ]:
with ProgressBar():
    gt_sst = mean_sst(ds_gt).load()
    pr_sst = mean_sst(ds_pr).load()

fig, _ = three_panel_maps(
    gt_sst,
    pr_sst,
    "OM4 SST (mean)",
    f"{MODEL_NAME} SST (mean)",
    cm.cm.thermal,
    suptitle="Sea surface temperature",
)
save_figure(fig, "sst_mean_maps")
plt.show()

### SSH (`zos`, mean over time)

In [ ]:
with ProgressBar():
    gt_z = mean_zos(ds_gt).load()
    pr_z = mean_zos(ds_pr).load()

fig, _ = three_panel_maps(
    gt_z,
    pr_z,
    "OM4 SSH (mean)",
    f"{MODEL_NAME} SSH (mean)",
    cm.cm.curl,
    suptitle="Sea surface height",
)
save_figure(fig, "ssh_mean_maps")
plt.show()

### Surface zonal velocity `uo` (mean over time)


In [ ]:
with ProgressBar():
    gt_uo = mean_surface_field(ds_gt, "uo", "u", "m/s").load()
    pr_uo = mean_surface_field(ds_pr, "uo", "u", "m/s").load()

fig, _ = three_panel_maps(
    gt_uo,
    pr_uo,
    "OM4 u (mean)",
    f"{MODEL_NAME} u (mean)",
    cm.cm.balance,
    suptitle="Surface zonal velocity",
)
save_figure(fig, "uo_surface_mean_maps")
plt.show()


### Surface meridional velocity `vo` (mean over time)


In [ ]:
with ProgressBar():
    gt_vo = mean_surface_field(ds_gt, "vo", "v", "m/s").load()
    pr_vo = mean_surface_field(ds_pr, "vo", "v", "m/s").load()

fig, _ = three_panel_maps(
    gt_vo,
    pr_vo,
    "OM4 v (mean)",
    f"{MODEL_NAME} v (mean)",
    cm.cm.balance,
    suptitle="Surface meridional velocity",
)
save_figure(fig, "vo_surface_mean_maps")
plt.show()


### Surface salinity `so` (mean over time)


In [ ]:
with ProgressBar():
    gt_so = mean_surface_field(ds_gt, "so", "S", "psu").load()
    pr_so = mean_surface_field(ds_pr, "so", "S", "psu").load()

fig, _ = three_panel_maps(
    gt_so,
    pr_so,
    "OM4 S (mean)",
    f"{MODEL_NAME} S (mean)",
    cm.cm.haline,
    suptitle="Sea surface salinity",
)
save_figure(fig, "so_surface_mean_maps")
plt.show()


### Optional: $\theta_O$ at one model depth level (mean over time)

Change `LEV_INDEX` to match a depth of interest (0 = surface).

In [ ]:
LEV_INDEX = 10  # edit

with ProgressBar():
    m = np.isnan(ds_gt["thetao"]).isel(lev=LEV_INDEX).isel(
        time=min(5, ds_gt.sizes["time"] - 1)
    )
    gt_t = ds_gt["thetao"].isel(lev=LEV_INDEX).mean("time").where(~m).load()
    pr_t = ds_pr["thetao"].isel(lev=LEV_INDEX).mean("time").where(~m).load()

lev_m = float(ds_gt["lev"].isel(lev=LEV_INDEX).values)
fig, _ = three_panel_maps(
    gt_t,
    pr_t,
    f"OM4 θ (mean), z≈{lev_m:.0f} m",
    f"{MODEL_NAME} θ (mean), z≈{lev_m:.0f} m",
    cm.cm.thermal,
    suptitle=f"Potential temperature at lev index {LEV_INDEX}",
)
save_figure(fig, f"theta_lev{LEV_INDEX}_mean_maps")
plt.show()
